In [0]:
# Project         : Procurement Analytics using Databricks & Power BI
# Layer           : Silver
# Notebook        : Silver_purchase_orders
# Source          : purchase_orders.csv
# Target          : procurement.silver.silver_purchase_orders
#
# Author          : V R Mutyala
# Created Date    : 21-Jul-2026
# Last Modified   : 21-Jul-2026
#
# Description
# -----------
# This notebook loads Cleaned purchase_orders master data into the Silver layer.
# It validates the source data, separates duplicate records, adds audit columns, and stores the results as Delta tables.

# ==============================================================================
# Business Objective
# ==============================================================================
#
# Read employee data from the Bronze layer, apply data cleansing,standardization, and business rules to create a trusted Silver Delta table.
# ==============================================================================

In [0]:
%run ../01_Config/Config

In [0]:
%run ../05_Helper_Functions/Helper_functions

In [0]:
# Import Libraries and Widgets
from pyspark.sql import DataFrame
from pyspark.sql.functions import (col, lit,current_timestamp,when,count,trim,coalesce,initcap,lower)
from pyspark.sql.types import (StructType, StructField, StringType,IntegerType,DoubleType,DecimalType)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number
from pyspark.sql.functions import try_to_timestamp

In [0]:
# ============================================================
# Read Bronze Invoice Table
# ============================================================

bronze_purchase_orders_df = read_delta(BRONZE_PURCHASE_ORDERS)

preview(bronze_purchase_orders_df,"Bronze purchase_orders")

In [0]:
#============================================
# Create Sliver DataFrame
#===========================================
silver_purchase_orders_df = bronze_purchase_orders_df

In [0]:
# ============================================================
# Apply business transformations
# ============================================================
silver_purchase_orders_df = (silver_purchase_orders_df

    # Trim string columns
    .withColumn("po_id", trim(col("po_id")))
    .withColumn("pr_id", trim(col("pr_id")))
    .withColumn("supplier_id", trim(col("supplier_id")))
    .withColumn("product_id", trim(col("product_id")))
    .withColumn("currency", upper(trim(col("currency"))))
    .withColumn("po_status", initcap(trim(col("po_status"))))

    # Date columns
    .withColumn("po_date",
                to_date(trim(col("po_date")), "dd-MM-yyyy"))
    .withColumn("expected_delivery_date",
                to_date(trim(col("expected_delivery_date")), "dd-MM-yyyy"))

    # Numeric columns (no trim)
    .withColumn("quantity_ordered", col("quantity_ordered").cast("int"))
    .withColumn("unit_price", col("unit_price").cast("decimal(10,2)"))
    .withColumn("total_amount", col("total_amount").cast("decimal(12,2)"))

    # Boolean column
    .withColumn("is_duplicate_po", col("is_duplicate_po").cast("boolean"))

    # Standardize payment terms
    .withColumn(
        "payment_terms",
        when(
            upper(trim(col("payment_terms"))).isin("NET30", "NET 30"),
            "Net 30"
        ).otherwise(initcap(trim(col("payment_terms"))))
    )
)


In [0]:
# ============================================================
# Identify Invalid Purchase Order Records
# NULL & Blank: po_id
# NULL: po_date, pr_id, supplier_id, product_id,
#       quantity_ordered, unit_price, total_amount,
#       currency, po_status, expected_delivery_date,
#       payment_terms
# Zero: quantity_ordered, unit_price, total_amount
# ============================================================

from pyspark.sql.functions import col, trim

invalid_purchase_orders = silver_purchase_orders_df.filter(
    col("po_id").isNull()
    | (trim(col("po_id")) == "")
    | col("po_date").isNull()
    | col("pr_id").isNull()
    | (trim(col("pr_id")) == "")
    | col("supplier_id").isNull()
    | (trim(col("supplier_id")) == "")
    | col("product_id").isNull()
    | (trim(col("product_id")) == "")
    | col("quantity_ordered").isNull()
    | (col("quantity_ordered") == 0)
    | col("unit_price").isNull()
    | (col("unit_price") == 0)
    | col("total_amount").isNull()
    | (col("total_amount") == 0)
    | col("currency").isNull()
    | (trim(col("currency")) == "")
    | col("po_status").isNull()
    | (trim(col("po_status")) == "")
    | col("expected_delivery_date").isNull()
    | col("payment_terms").isNull()
    | (trim(col("payment_terms")) == "")
)

print(f"Invalid Purchase Order Records: {invalid_purchase_orders.count()}")

display(invalid_purchase_orders)

In [0]:
# ============================================================
# Add Audit Metadata
# ============================================================

invalid_purchase_orders = (invalid_purchase_orders.withColumn("audit_timestamp",current_timestamp())
    .withColumn("source_table",lit("purchase_orders"))
    .withColumn("pipeline_layer",lit("Silver"))
    .withColumn("issue_type",lit("Invalid Record")))

display(invalid_purchase_orders)

In [0]:
# ============================================================
# Write Invalid PURCHASE_ORDERS to Audit Table
# ============================================================

if invalid_purchase_orders.count() > 0:
    write_delta(invalid_purchase_orders,AUDIT_INVALID_PURCHASE_ORDERS,mode="overwrite")
    print("Invalid invoice records written.")
else:
    print("No invalid invoice records found.")

In [0]:
# ============================================================
# Remove Invalid Records
# ============================================================

from pyspark.sql.functions import col, trim

silver_purchase_orders_df = silver_purchase_orders_df.filter(

    col("po_id").isNotNull() &
    (trim(col("po_id")) != "") &

    col("po_date").isNotNull() &

    col("pr_id").isNotNull() &
    (trim(col("pr_id")) != "") &

    col("supplier_id").isNotNull() &
    (trim(col("supplier_id")) != "") &

    col("product_id").isNotNull() &
    (trim(col("product_id")) != "") &

    col("quantity_ordered").isNotNull() &
    (col("quantity_ordered") != 0) &

    col("unit_price").isNotNull() &
    (col("unit_price") != 0) &

    col("total_amount").isNotNull() &
    (col("total_amount") != 0) &

    col("currency").isNotNull() &
    (trim(col("currency")) != "") &

    col("po_status").isNotNull() &
    (trim(col("po_status")) != "") &

    col("is_duplicate_po").isNotNull() &

    col("expected_delivery_date").isNotNull() &

    col("payment_terms").isNotNull() &
    (trim(col("payment_terms")) != "")
)

display(silver_purchase_orders_df)

In [0]:
# ============================================================
# Remove Duplicate PURCHASE_ORDERS
# ============================================================

window_spec = Window.partitionBy("po_id").orderBy("po_date")

silver_purchase_orders_df = (silver_purchase_orders_df.withColumn("row_num",row_number().over(window_spec))
                      .filter(col("row_num") == 1).drop("row_num"))
preview(silver_purchase_orders_df,"Silver Purchase_Orders")


In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

silver_purchase_orders_df = (silver_purchase_orders_df
                         .withColumn("silver_load_timestamp", current_timestamp())
                         .withColumn("pipeline_layer", lit("Silver"))
)

In [0]:
# ============================================================
# Write Silver Delta Table
# ============================================================

write_delta(df=silver_purchase_orders_df,table_name=SILVER_PURCHASE_ORDERS)

In [0]:
# ============================================================
# Validate Summary
# ============================================================

bronze_count = bronze_purchase_orders_df.count()
invalid_count = invalid_purchase_orders.count()
silver_count = silver_purchase_orders_df.count()

duplicate_removed = bronze_count - invalid_count - silver_count

print("=" * 60)
print("Silver purchase_orders Load Completed Successfully")
print("=" * 60)

print(f"Bronze Records            : {bronze_count}")
print(f"Invalid Records Removed   : {invalid_count}")
print(f"Duplicate Records Removed : {duplicate_removed}")
print(f"Silver Records            : {silver_count}")